# **Notebook 07 — Gemma 4 + RAG V1**

**Module:** 2 — Telecom RAG V1  
**System:** Gemma 4 + frozen RAG V1 via vLLM  
**Purpose:** Evaluate Gemma 4 with the same BGE-M3/FAISS Top-7 retrieval architecture used by the other RAG configuration, enabling direct comparison with Gemma 4 LLM-only.


# **1. RAG V1 Environment Initialisation**

## **1.1. Load RAG Dependencies**

### **Install All Required Libraries**

In [4]:
# =============================================================================
# Retriever Implementation
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    pipeline \
    tqdm

print("All required Module 2 libraries installed successfully.")

All required Module 2 libraries installed successfully.


In [5]:
!pip install sentence-transformers --upgrade --no-deps

### **Import All Required Libraries**

In [6]:
# =============================================================================
# Retriever Implementation
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.13.0+cu130
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Import Datasets from Kaggle**

In [7]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [9]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_chunks_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-reconciled-chunks')
cliffordimaguezegie_telecom_bge_m3_embeddings_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-embeddings')
cliffordimaguezegie_telecom_bge_m3_faiss_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-faiss')

print('Data source import complete.')


Data source import complete.


In [10]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')

print('Data source import complete.')


Data source import complete.


In [11]:
print("Chunks:")
print(cliffordimaguezegie_telecom_chunks_path)

print("\nEmbeddings:")
print(cliffordimaguezegie_telecom_bge_m3_embeddings_path)

print("\nFAISS:")
print(cliffordimaguezegie_telecom_bge_m3_faiss_path)

print("\nQUESTIONS:")
print(cliffordimaguezegie_telecom_benchmark_path)




Chunks:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1

Embeddings:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1

FAISS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [12]:
from pathlib import Path

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

EMBED_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_embeddings_path
)

FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

BENCHMARK_DIR = Path(
    cliffordimaguezegie_telecom_benchmark_path
)

print("Chunks    :", CHUNK_DIR)
print("Embeddings:", EMBED_DIR)
print("FAISS     :", FAISS_DIR)
print("BENCHMARK :", BENCHMARK_DIR)

Chunks    : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1
Embeddings: /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1
FAISS     : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1
BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [13]:
# =============================================================================
# RAG V1 — INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

RAG V1 — BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [14]:
# =============================================================================
# RAG V1 — LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

RAG V1 — BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


### **GPU Verification**

In [15]:
# =============================================================================
# GPU (CORRECTED WITH DRIVER-LEVEL MEMORY INFO)
# =============================================================================

print("\nGPU / VRAM")
print("-" * 40)

device_count = torch.cuda.device_count()
print(f"GPU count       : {device_count}")

if device_count == 0:
    print("⚠️ No CUDA GPUs detected!")
else:
    for gpu_id in range(device_count):
        props = torch.cuda.get_device_properties(gpu_id)

        # True driver-level memory query (handles external processes like vLLM)
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)

        total_gpu_gb = total_bytes / (1024**3)
        free_gpu_gb = free_bytes / (1024**3)
        used_gpu_gb = total_gpu_gb - free_gpu_gb

        # PyTorch specific tracking (optional breakdown)
        torch_allocated_gb = torch.cuda.memory_allocated(gpu_id) / (1024**3)
        torch_reserved_gb = torch.cuda.memory_reserved(gpu_id) / (1024**3)

        print(f"\nGPU {gpu_id}: {props.name}")
        print(f"  Total VRAM     : {total_gpu_gb:.2f} GB")
        print(f"  Used VRAM (All): {used_gpu_gb:.2f} GB")
        print(f"  Free VRAM      : {free_gpu_gb:.2f} GB")
        print(f"  --- PyTorch Internal Tracking ---")
        print(f"  Allocated      : {torch_allocated_gb:.2f} GB")
        print(f"  Reserved       : {torch_reserved_gb:.2f} GB")


GPU / VRAM
----------------------------------------
GPU count       : 1

GPU 0: NVIDIA L4
  Total VRAM     : 22.03 GB
  Used VRAM (All): 16.20 GB
  Free VRAM      : 5.84 GB
  --- PyTorch Internal Tracking ---
  Allocated      : 0.00 GB
  Reserved       : 0.00 GB


## **1.2. Load Retriever V1**

### **Load Retriever**

In [16]:
# =============================================================================
# RETRIEVER V1 — STANDARD IN-MEMORY IMPLEMENTATION
# =============================================================================

import json
import time
from pathlib import Path

import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer


# =============================================================================
# RETRIEVER V1 CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

MAX_SEQ_LENGTH = 1280

DEFAULT_K = 7

EXPECTED_VECTORS = 1_506_367

EXPECTED_DIMENSION = 1024

RETRIEVER_VERSION = "V1"


# =============================================================================
# RUNTIME OBJECTS
# =============================================================================

faiss_index = None

query_encoder = None

vector_metadata = None

chunk_text_map = None

faiss_manifest = None


# =============================================================================
# INITIALIZE RETRIEVER
# =============================================================================

def initialize_retriever(
    faiss_dir,
    chunk_dir,
    device="cuda",
):
    """
    Initialize Retriever V1.

    Loads:
        - FAISS index
        - BGE-M3 query encoder
        - Vector metadata
        - Chunk text
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map
    global faiss_manifest


    # =========================================================================
    # VALIDATE DEVICE
    # =========================================================================

    if device.startswith("cuda"):

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA GPU is required for Retriever V1."
            )


    # =========================================================================
    # RESOLVE PATHS
    # =========================================================================

    faiss_dir = Path(
        faiss_dir
    )

    chunk_dir = Path(
        chunk_dir
    )


    # =========================================================================
    # FAISS ARTIFACTS
    # =========================================================================

    faiss_index_path = (
        faiss_dir
        / "faiss_index_flat_ip.index"
    )

    faiss_manifest_path = (
        faiss_dir
        / "faiss_manifest.json"
    )


    for path in [
        faiss_dir,
        chunk_dir,
        faiss_index_path,
        faiss_manifest_path,
    ]:

        if not path.exists():

            raise FileNotFoundError(
                f"Required Retriever V1 resource not found:\n"
                f"{path}"
            )


    # =========================================================================
    # LOAD FAISS
    # =========================================================================

    faiss_index = faiss.read_index(
        str(faiss_index_path)
    )


    if faiss_index.ntotal != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Unexpected FAISS vector count: "
            f"{faiss_index.ntotal:,}"
        )


    if faiss_index.d != EXPECTED_DIMENSION:

        raise RuntimeError(
            f"Unexpected FAISS dimension: "
            f"{faiss_index.d}"
        )


    # =========================================================================
    # LOAD FAISS MANIFEST
    # =========================================================================

    with open(
        faiss_manifest_path,
        "r",
        encoding="utf-8",
    ) as file:

        faiss_manifest = json.load(
            file
        )


    # =========================================================================
    # LOAD BGE-M3
    # =========================================================================

    query_encoder = SentenceTransformer(
        MODEL_NAME,
        device=device,
    )

    query_encoder.max_seq_length = (
        MAX_SEQ_LENGTH
    )

    query_encoder.half()

    query_encoder.eval()


    # =========================================================================
    # LOAD VECTOR METADATA
    # =========================================================================

    vector_mapping_path = (
        faiss_dir
        / "vector_mapping.jsonl"
    )

    if not vector_mapping_path.exists():

        raise FileNotFoundError(
            f"Vector mapping not found:\n"
            f"{vector_mapping_path}"
        )


    vector_metadata = {}


    with open(
        vector_mapping_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            record = json.loads(
                line
            )

            vector_id = int(
                record["vector_id"]
            )

            vector_metadata[
                vector_id
            ] = record


    if len(vector_metadata) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"metadata records, found "
            f"{len(vector_metadata):,}"
        )


    # =========================================================================
    # LOAD CHUNK TEXT
    # =========================================================================

    chunk_shards = sorted(
        chunk_dir.glob(
            "chunks_*.jsonl"
        )
    )


    if not chunk_shards:

        raise FileNotFoundError(
            "No chunk shards found in:\n"
            f"{chunk_dir}"
        )


    chunk_text_map = {}


    for shard_path in chunk_shards:

        with open(
            shard_path,
            "r",
            encoding="utf-8",
        ) as file:

            for line in file:

                if not line.strip():
                    continue

                record = json.loads(
                    line
                )

                chunk_id = record.get(
                    "chunk_id"
                )

                if chunk_id is not None:

                    chunk_text_map[
                        chunk_id
                    ] = record["text"]


    if len(chunk_text_map) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"chunk texts, found "
            f"{len(chunk_text_map):,}"
        )


    # =========================================================================
    # RETURN STATUS
    # =========================================================================

    return {
        "version": RETRIEVER_VERSION,
        "model": MODEL_NAME,
        "device": device,
        "embedding_dimension":
            EXPECTED_DIMENSION,
        "faiss_vectors":
            int(faiss_index.ntotal),
        "faiss_index":
            faiss_manifest.get(
                "index_type"
            ),
        "similarity":
            faiss_manifest.get(
                "similarity"
            ),
        "metadata_rows":
            len(vector_metadata),
        "chunk_rows":
            len(chunk_text_map),
        "default_k":
            DEFAULT_K,
    }


# =============================================================================
# RETRIEVE
# =============================================================================

def retrieve(
    query: str,
    k: int = DEFAULT_K,
):
    """
    Execute Retriever V1.
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map


    # =========================================================================
    # VALIDATE INITIALIZATION
    # =========================================================================

    if (
        faiss_index is None
        or query_encoder is None
        or vector_metadata is None
        or chunk_text_map is None
    ):

        raise RuntimeError(
            "Retriever V1 is not initialized. "
            "Call initialize_retriever() first."
        )


    # =========================================================================
    # INPUT VALIDATION
    # =========================================================================

    if not isinstance(
        query,
        str,
    ):

        raise TypeError(
            "query must be a string."
        )


    query = query.strip()


    if not query:

        raise ValueError(
            "query cannot be empty."
        )


    if not isinstance(
        k,
        int,
    ):

        raise TypeError(
            "k must be an integer."
        )


    if k <= 0:

        raise ValueError(
            "k must be greater than zero."
        )


    # =========================================================================
    # QUERY EMBEDDING
    # =========================================================================

    embedding_start = time.time()


    query_embedding = (
        query_encoder.encode(
            [query],
            batch_size=1,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )


    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )


    query_embedding /= np.linalg.norm(
        query_embedding,
        axis=1,
        keepdims=True,
    )


    embedding_time = (
        time.time()
        - embedding_start
    )


    # =========================================================================
    # FAISS SEARCH
    # =========================================================================

    search_start = time.time()


    scores, vector_ids = (
        faiss_index.search(
            query_embedding,
            k,
        )
    )


    search_time = (
        time.time()
        - search_start
    )


    # =========================================================================
    # RESULT RESOLUTION
    # =========================================================================

    results = []


    for rank, (
        vector_id,
        score,
    ) in enumerate(
        zip(
            vector_ids[0],
            scores[0],
        ),
        start=1,
    ):

        vector_id = int(
            vector_id
        )


        if vector_id < 0:
            continue


        metadata = vector_metadata.get(
            vector_id
        )


        if metadata is None:

            continue


        chunk_id = metadata.get(
            "chunk_id"
        )


        text = chunk_text_map.get(
            chunk_id
        )


        if text is None:

            continue


        results.append(
            {
                "rank": rank,
                "score": float(score),
                "vector_id": vector_id,
                "chunk_id": chunk_id,
                "document_id":
                    metadata.get(
                        "document_id"
                    ),
                "source":
                    metadata.get(
                        "source"
                    ),
                "title":
                    metadata.get(
                        "title"
                    ),
                "path":
                    metadata.get(
                        "path"
                    ),
                "text": text,
            }
        )


    # =========================================================================
    # TOTAL LATENCY
    # =========================================================================

    total_time = (
        embedding_time
        + search_time
    )


    return {
        "query": query,
        "k": k,
        "results": results,
        "timing": {
            "query_embedding_sec":
                embedding_time,
            "faiss_search_sec":
                search_time,
            "total_sec":
                total_time,
        },
    }

### **Initialize Retriever**

In [17]:
FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

retriever_status = initialize_retriever(
    faiss_dir=FAISS_DIR,
    chunk_dir=CHUNK_DIR,
    device="cuda",
)

print("=" * 90)
print("RETRIEVER V1 INITIALIZED")
print("=" * 90)

for key, value in retriever_status.items():
    print(
        f"{key:<25}: {value}"
    )

print("=" * 90)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RETRIEVER V1 INITIALIZED
version                  : V1
model                    : BAAI/bge-m3
device                   : cuda
embedding_dimension      : 1024
faiss_vectors            : 1506367
faiss_index              : IndexFlatIP
similarity               : inner_product
metadata_rows            : 1506367
chunk_rows               : 1506367
default_k                : 7


### **Validate Retriever**

In [18]:
# =============================================================================
# RAG V1 — RETRIEVER V1 FINAL VALIDATION
# =============================================================================

TEST_QUERY = (
    "What are the primary responsibilities of the AMF "
    "in a 5G Standalone network?"
)

results = retrieve(
    TEST_QUERY,
    k=7,
)

print("=" * 90)
print("RAG V1 — RETRIEVER V1 FINAL VALIDATION")
print("=" * 90)

print(
    f"Query              : "
    f"{results['query']}"
)

print(
    f"K                  : "
    f"{results['k']}"
)

print(
    f"Results returned   : "
    f"{len(results['results'])}"
)

print(
    f"Query embedding    : "
    f"{results['timing']['query_embedding_sec']:.4f} sec"
)

print(
    f"FAISS search       : "
    f"{results['timing']['faiss_search_sec']:.4f} sec"
)

print(
    f"Total retrieval    : "
    f"{results['timing']['total_sec']:.4f} sec"
)

if len(results["results"]) != 7:
    raise RuntimeError(
        "Retriever V1 did not return 7 results."
    )

top = results["results"][0]

print("\nTop result:")

print(
    f"Rank       : {top['rank']}"
)

print(
    f"Score      : {top['score']:.4f}"
)

print(
    f"Chunk ID   : {top['chunk_id']}"
)

print(
    f"Source     : {top['source']}"
)

print(
    f"Title      : {top['title']}"
)

print(
    f"Text       : "
    f"{top['text'][:500]}"
)

print("\n" + "=" * 90)
print("RETRIEVER V1 FINAL VALIDATION PASSED")
print("=" * 90)

RAG V1 — RETRIEVER V1 FINAL VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
K                  : 7
Results returned   : 7
Query embedding    : 0.7271 sec
FAISS search       : 0.2550 sec
Total retrieval    : 0.9821 sec

Top result:
Rank       : 1
Score      : 0.7065
Chunk ID   : standards/3gpp_rel18/original/rel_15.docx::chunk_0013
Source     : standards
Title      : rel_15
Text       : the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), takes care of the signalling related to User Data traffic, such as session establishment. Finally, The UPF ("User Plane Function") represents the handling of user data.
On the Access Network side, the gNB (5G Node B) performs all the main AN-related tasks, including Radio Resource Manageme

RETRIEVER V1 FINAL VALIDATION PASSED


## **1.3. Load Gemma 4 LLM**

In [36]:
import subprocess

# Terminate any lingering vLLM background process if it exists
try:
    vllm_process.terminate()
    vllm_process.wait()
    print("🛑 Previous vLLM process terminated.")
except NameError:
    # If vllm_process isn't defined in this scope yet, use shell command to kill port 8000
    subprocess.run(["pkill", "-f", "vllm"], stderr=subprocess.DEVNULL)
    print("🧹 Cleared any residual vLLM processes.")

# Clear torch CUDA cache just in case
import torch
torch.cuda.empty_cache()
print("✨ VRAM cache cleared!")

🛑 Previous vLLM process terminated.
✨ VRAM cache cleared!


In [2]:
# =============================================================================
# GEMMA 4 — vLLM QAT W4A16 LOADING
# =============================================================================

# 1. Install packages
!pip install vllm openai nest_asyncio requests hf_transfer huggingface_hub --upgrade

# TorchAudio is only required if your environment needs the matching
# PyTorch CUDA 13.0 nightly build.
!pip install --upgrade torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu130 \
    --force-reinstall


# =============================================================================
# 2. HUGGING FACE AUTHENTICATION
# =============================================================================

from huggingface_hub import notebook_login, get_token

import os
import subprocess
import time
import requests
import nest_asyncio

from openai import OpenAI


notebook_login()

hf_token = get_token()

if not hf_token:
    raise ValueError(
        "⚠️ No Hugging Face token found! "
        "Please make sure notebook_login() was successful."
    )


# =============================================================================
# 3. ENVIRONMENT CONFIGURATION
# =============================================================================

env = os.environ.copy()

env["HF_TOKEN"] = hf_token

env["HF_XET_HIGH_PERFORMANCE"] = "1"

env["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "1800"
env["VLLM_WORKER_TIMEOUT_S"] = "1800"

nest_asyncio.apply()


# =============================================================================
# 4. MODEL CONFIGURATION
# =============================================================================

VLLM_MODEL = "google/gemma-4-12B-it-qat-w4a16-ct"

VLLM_PORT = "8000"

print(
    f"⏳ Launching {VLLM_MODEL} "
    f"on vLLM server with Google QAT W4A16..."
)


# =============================================================================
# 5. vLLM SERVER COMMAND
# =============================================================================

vllm_command = [
    "vllm",
    "serve",
    VLLM_MODEL,

    "--port",
    VLLM_PORT,

    "--tensor-parallel-size",
    "1",

    "--max-model-len",
    "10240",

    "--gpu-memory-utilization",
    "0.75",

    "--limit-mm-per-prompt",
    '{"image": 0, "audio": 0}',
]


print("\nLaunching vLLM with command:")
print(" ".join(vllm_command))


# =============================================================================
# 6. START vLLM SERVER
# =============================================================================

log_file = open(
    "vllm_server.log",
    "w"
)

vllm_process = subprocess.Popen(
    vllm_command,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)


# =============================================================================
# 7. WAIT FOR SERVER
# =============================================================================

max_attempts = 90

attempt = 0
server_ready = False

while attempt < max_attempts:

    # Check whether vLLM crashed during startup
    if vllm_process.poll() is not None:

        print(
            "\n❌ vLLM process terminated unexpectedly!"
        )

        print(
            "\nLast 30 lines of server log:"
        )

        with open(
            "vllm_server.log",
            "r"
        ) as f:

            lines = f.readlines()

            print(
                "".join(lines[-30:])
            )

        raise RuntimeError(
            "vLLM Server crashed during initialization."
        )


    # Check API availability
    try:

        response = requests.get(
            "http://localhost:8000/v1/models"
        )

        if response.status_code == 200:

            print(
                "\n✅ vLLM Server is successfully "
                "running and ready!"
            )

            server_ready = True

            break

    except requests.exceptions.ConnectionError:

        pass


    attempt += 1

    time.sleep(10)

    print(
        f"⏳ Loading QAT weights & booting engine... "
        f"(Attempt {attempt}/{max_attempts})"
    )


# =============================================================================
# 8. SERVER TIMEOUT
# =============================================================================

if not server_ready:

    vllm_process.terminate()

    raise RuntimeError(
        "❌ vLLM server timed out. "
        "Check vllm_server.log for details."
    )


# =============================================================================
# 9. OPENAI-COMPATIBLE CLIENT
# =============================================================================

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)


# =============================================================================
# 10. TEST GEMMA 4
# =============================================================================

response = client.chat.completions.create(
    model=VLLM_MODEL,

    messages=[
        {
            "role": "system",
            "content": (
                "You are an expert telecommunications "
                "network engineer."
            ),
        },
        {
            "role": "user",
            "content": (
                "Explain how BGP route reflection "
                "works in large networks."
            ),
        },
    ],

    temperature=0.01,
    max_tokens=500,
)


# =============================================================================
# 11. DISPLAY RESPONSE
# =============================================================================

print("\n" + "=" * 90)
print("GEMMA 4 QAT W4A16 TEST RESPONSE")
print("=" * 90)

print(
    response.choices[0].message.content
)

print("=" * 90)

  Using cached torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl (1.8 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0.dev20260826+cu130
    Uninstalling torchaudio-2.11.0.dev20260826+cu130:
      Successfully uninstalled torchaudio-2.11.0.dev20260826+cu130
Looking in indexes: https://download.pytorch.org/whl/nightly/cu130
  Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260826%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (7.4 kB)
Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260826%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0
    Uninstalling torchaudio-2.11.0:
      Successfully uninstalled torchaudio-2.11.0
ERROR: pip's dependency resolver does not currently take into a

## **1.4. Post GPU Verification**

In [19]:
# =============================================================================
# GPU (CORRECTED WITH DRIVER-LEVEL MEMORY INFO)
# =============================================================================

print("\nGPU / VRAM")
print("-" * 40)

device_count = torch.cuda.device_count()
print(f"GPU count       : {device_count}")

if device_count == 0:
    print("⚠️ No CUDA GPUs detected!")
else:
    for gpu_id in range(device_count):
        props = torch.cuda.get_device_properties(gpu_id)

        # True driver-level memory query (handles external processes like vLLM)
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)

        total_gpu_gb = total_bytes / (1024**3)
        free_gpu_gb = free_bytes / (1024**3)
        used_gpu_gb = total_gpu_gb - free_gpu_gb

        # PyTorch specific tracking (optional breakdown)
        torch_allocated_gb = torch.cuda.memory_allocated(gpu_id) / (1024**3)
        torch_reserved_gb = torch.cuda.memory_reserved(gpu_id) / (1024**3)

        print(f"\nGPU {gpu_id}: {props.name}")
        print(f"  Total VRAM     : {total_gpu_gb:.2f} GB")
        print(f"  Used VRAM (All): {used_gpu_gb:.2f} GB")
        print(f"  Free VRAM      : {free_gpu_gb:.2f} GB")
        print(f"  --- PyTorch Internal Tracking ---")
        print(f"  Allocated      : {torch_allocated_gb:.2f} GB")
        print(f"  Reserved       : {torch_reserved_gb:.2f} GB")


GPU / VRAM
----------------------------------------
GPU count       : 1

GPU 0: NVIDIA L4
  Total VRAM     : 22.03 GB
  Used VRAM (All): 18.85 GB
  Free VRAM      : 3.18 GB
  --- PyTorch Internal Tracking ---
  Allocated      : 1.07 GB
  Reserved       : 2.61 GB


# **2. Context Assembly**

In [20]:
# =============================================================================
# RAG V1 — CONTEXT ASSEMBLY (REFACTORED)
# =============================================================================

def assemble_context(retrieval_results: list) -> str:
    """
    Convert Retriever V1 results into an LLM-ready context block.

    Preserves retrieval rank, source, title, chunk ID, and text
    while remaining resilient to missing metadata.
    """
    if not retrieval_results:
        return "NO RELEVANT CONTEXT FOUND."

    context_blocks = []

    for idx, result in enumerate(retrieval_results, start=1):
        # Extract metadata safely to prevent KeyError
        rank = result.get("rank", idx)
        source = result.get("source", "Unknown Source")
        title = result.get("title", "Untitled Document")
        chunk_id = result.get("chunk_id", "N/A")
        text = result.get("text", "").strip()

        # Clean block structure optimized for LLM attention
        block = (
            f"--- START DOCUMENT [{rank}] ---\n"
            f"Title: {title}\n"
            f"Source: {source}\n"
            f"Chunk ID: {chunk_id}\n\n"
            f"{text}\n"
            f"--- END DOCUMENT [{rank}] ---"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)


# =============================================================================
# TEST CONTEXT ASSEMBLY
# =============================================================================

TEST_QUERY = "What are the primary responsibilities of the AMF in a 5G Standalone network?"

retrieval_output = retrieve(TEST_QUERY, k=7)
rag_context = assemble_context(retrieval_output.get("results", []))

# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — CONTEXT ASSEMBLY VALIDATION")
print("=" * 90)

print(f"Query              : {retrieval_output.get('query', TEST_QUERY)}")
print(f"Retrieved chunks   : {len(retrieval_output.get('results', []))}")
print(f"Context characters : {len(rag_context):,}")

print("\nContext preview:")
print("-" * 90)
print(rag_context[:3000])

print("\n" + "=" * 90)
print("CONTEXT ASSEMBLY VALIDATED")
print("=" * 90)

RAG V1 — CONTEXT ASSEMBLY VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
Retrieved chunks   : 7
Context characters : 20,665

Context preview:
------------------------------------------------------------------------------------------
--- START DOCUMENT [1] ---
Title: rel_15
Source: standards
Chunk ID: standards/3gpp_rel18/original/rel_15.docx::chunk_0013

the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), takes care of the signalling related to User Data traffic, such as session establishment. Finally, The UPF ("User Plane Function") represents the handling of user data.
On the Access Network side, the gNB (5G Node B) performs all the main AN-related tasks, including Radio Resource Management: Radio Bearer Control, Radio Admission Control, Connection Mobility Control, D

# **3. Prompt Construction**

### **Prompt Definition**

In [21]:
# =============================================================================
# TELECOM RAG V1 — HARDENED SYSTEM PROMPT
# =============================================================================

RAG_SYSTEM_PROMPT = """
You are an expert telecom engineering assistant.

Your task is to answer the user's question using the provided telecom document context.

Rules:
1. Base the answer EXCLUSIVELY on the provided context. Do not invent, assume, or extrapolate technical facts beyond what is written.
2. Do not use outside knowledge to fill in missing details.
3. If the context does not contain enough information to answer the question, respond EXACTLY with: "The requested details are not available in the documentation."
4. Answer only what is directly relevant to the user's question.
5. Do not restate or repeat the user's question.
6. Avoid repetition or rephrasing the same technical points.
7. Organize multi-step functions, responsibilities, or architectural components using clear bullet points or short structured sub-headings.
8. Provide only the final answer—do not reveal internal reasoning, scratchpads, or chain-of-thought.
9. Do not mention "context", "documents", "retrieval", "sources", or "prompts".
10. NEVER use meta-introductory phrases such as "Based on the provided context", "According to the documentation", or "The text states".
""".strip()

# =============================================================================
# RAG CHAT MESSAGE BUILDER
# =============================================================================

def build_rag_messages(
    query: str,
    context: str,
) -> list[dict[str, str]]:
    """
    Build standardized chat messages for RAG generation.

    Handles empty or whitespace-only context gracefully without crashing.
    """
    if not isinstance(query, str):
        raise TypeError("query must be a string.")

    if not isinstance(context, str):
        raise TypeError("context must be a string.")

    clean_query = query.strip()
    clean_context = context.strip()

    if not clean_query:
        raise ValueError("query cannot be empty or whitespace-only.")

    # Gracefully handle empty retrieval results without raising an exception
    if not clean_context:
        clean_context = "NO CONTEXT AVAILABLE."

    user_content = f"""
### Context

{clean_context}

### Question

{clean_query}

Provide a concise, technically accurate answer following all system rules.
""".strip()

    return [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

### **Prompt Validation**

In [22]:
# =============================================================================
# RAG V1 — PROMPT MESSAGE VALIDATION (ENHANCED)
# =============================================================================

# 1. Execute Pipeline Steps Safely
retrieval_output = retrieve(TEST_QUERY, k=7)
retrieved_chunks = retrieval_output.get("results", [])

rag_context = assemble_context(retrieved_chunks)

rag_messages = build_rag_messages(
    query=TEST_QUERY,
    context=rag_context,
)

# 2. Extract Message Metrics
system_msg = rag_messages[0]["content"]
user_msg = rag_messages[1]["content"]

# Standard rule of thumb: ~4 characters per token for English text
est_user_tokens = len(user_msg) // 4
est_system_tokens = len(system_msg) // 4
est_total_tokens = est_user_tokens + est_system_tokens

# 3. Print Validation Telemetry
print("=" * 90)
print("RAG V1 — RAG MESSAGE VALIDATION")
print("=" * 90)

print(f"Query              : {TEST_QUERY}")
print(f"Retrieved Chunks   : {len(retrieved_chunks)}")
print(f"Context Characters : {len(rag_context):,}")
print(f"Message Count      : {len(rag_messages)}")
print(f"Est. Total Tokens  : ~{est_total_tokens:,} (System: ~{est_system_tokens}, User: ~{est_user_tokens})")

print("\nSystem Message:")
print("-" * 90)
print(system_msg)

print("\nUser Message Preview (First 3,000 chars):")
print("-" * 90)
print(user_msg[:3000])

if len(user_msg) > 3000:
    print(f"\n... [Truncated {len(user_msg) - 3000:,} remaining characters from preview]")

print("\n" + "=" * 90)
print("RAG MESSAGE VALIDATION COMPLETE")
print("=" * 90)

RAG V1 — RAG MESSAGE VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
Retrieved Chunks   : 7
Context Characters : 20,665
Message Count      : 2
Est. Total Tokens  : ~5,495 (System: ~284, User: ~5211)

System Message:
------------------------------------------------------------------------------------------
You are an expert telecom engineering assistant.

Your task is to answer the user's question using the provided telecom document context.

Rules:
1. Base the answer EXCLUSIVELY on the provided context. Do not invent, assume, or extrapolate technical facts beyond what is written.
2. Do not use outside knowledge to fill in missing details.
3. If the context does not contain enough information to answer the question, respond EXACTLY with: "The requested details are not available in the documentation."
4. Answer only what is directly relevant to the user's question.
5. Do not restate or repeat the user's question.
6. Avoid repet

### **Model-Native Chat Template Validation**

In [23]:
# =============================================================================
# RAG V1 — vLLM CHAT TEMPLATE / MESSAGE VALIDATION
# =============================================================================

# IMPORTANT:
# vLLM owns the Gemma 4 tokenizer and chat template.
# Therefore, we do NOT call apply_chat_template() locally.
#
# This validation confirms that the RAG messages produced by
# build_rag_messages() are accepted by the vLLM OpenAI-compatible
# Chat Completions API.

print("=" * 90)
print("RAG V1 — vLLM CHAT TEMPLATE / MESSAGE VALIDATION")
print("=" * 90)

print(f"Message Count      : {len(rag_messages)}")

print("\nMessage Roles:")
for idx, message in enumerate(rag_messages, start=1):
    print(
        f"  Message {idx}: "
        f"role={message.get('role')}, "
        f"characters={len(message.get('content', '')):,}"
    )

# -------------------------------------------------------------------------
# Local character telemetry
# -------------------------------------------------------------------------

system_msg = rag_messages[0]["content"]
user_msg = rag_messages[1]["content"]

print(f"\nSystem Characters  : {len(system_msg):,}")
print(f"User Characters    : {len(user_msg):,}")
print(f"Total Characters   : {len(system_msg) + len(user_msg):,}")

# Rough token estimate for logging only.
# Actual Gemma 4 tokenization is performed by vLLM.
est_system_tokens = len(system_msg) // 4
est_user_tokens = len(user_msg) // 4
est_total_tokens = est_system_tokens + est_user_tokens

print(
    f"Est. Total Tokens  : ~{est_total_tokens:,} "
    f"(System: ~{est_system_tokens:,}, "
    f"User: ~{est_user_tokens:,})"
)

# -------------------------------------------------------------------------
# Validate message structure before sending to vLLM
# -------------------------------------------------------------------------

message_validation_pass = True

for idx, message in enumerate(rag_messages, start=1):

    if not isinstance(message, dict):
        message_validation_pass = False
        print(f"\n[ERROR] Message {idx} is not a dictionary.")

    elif "role" not in message:
        message_validation_pass = False
        print(f"\n[ERROR] Message {idx} is missing 'role'.")

    elif "content" not in message:
        message_validation_pass = False
        print(f"\n[ERROR] Message {idx} is missing 'content'.")

    elif not isinstance(message["content"], str):
        message_validation_pass = False
        print(f"\n[ERROR] Message {idx} content is not a string.")

# -------------------------------------------------------------------------
# Display system message
# -------------------------------------------------------------------------

print("\nSystem Message:")
print("-" * 90)
print(system_msg)

# -------------------------------------------------------------------------
# Display user message preview
# -------------------------------------------------------------------------

print("\nUser Message Preview (First 3,000 chars):")
print("-" * 90)
print(user_msg[:3000])

if len(user_msg) > 3000:
    print(
        f"\n... [Truncated "
        f"{len(user_msg) - 3000:,} remaining characters from preview]"
    )

# -------------------------------------------------------------------------
# Final validation
# -------------------------------------------------------------------------

print("\n" + "=" * 90)

if message_validation_pass:
    print("MESSAGE STRUCTURE VALIDATED")
    print("Ready for vLLM Chat Completions API")
else:
    print("MESSAGE STRUCTURE VALIDATION FAILED")

print("=" * 90)

RAG V1 — vLLM CHAT TEMPLATE / MESSAGE VALIDATION
Message Count      : 2

Message Roles:
  Message 1: role=system, characters=1,136
  Message 2: role=user, characters=20,846

System Characters  : 1,136
User Characters    : 20,846
Total Characters   : 21,982
Est. Total Tokens  : ~5,495 (System: ~284, User: ~5,211)

System Message:
------------------------------------------------------------------------------------------
You are an expert telecom engineering assistant.

Your task is to answer the user's question using the provided telecom document context.

Rules:
1. Base the answer EXCLUSIVELY on the provided context. Do not invent, assume, or extrapolate technical facts beyond what is written.
2. Do not use outside knowledge to fill in missing details.
3. If the context does not contain enough information to answer the question, respond EXACTLY with: "The requested details are not available in the documentation."
4. Answer only what is directly relevant to the user's question.
5. Do not

# **4. End-to-End Gemma 4 + RAG Generation**

## **Generation Configuration Definition**

In [24]:
# =============================================================================
# RAG V1 — vLLM GENERATION CONFIGURATION
# =============================================================================

VLLM_GENERATION_CONFIG = {
    "temperature": 0.01,
    "top_p": 0.95,
    "max_tokens": 1024,
}

print("=" * 90)
print("RAG V1 — vLLM GENERATION CONFIGURATION")
print("=" * 90)

print(f"Temperature        : {VLLM_GENERATION_CONFIG['temperature']}")
print(f"Top-p              : {VLLM_GENERATION_CONFIG['top_p']}")
print(f"Max tokens         : {VLLM_GENERATION_CONFIG['max_tokens']}")

print("=" * 90)

RAG V1 — vLLM GENERATION CONFIGURATION
Temperature        : 0.01
Top-p              : 0.95
Max tokens         : 1024


## **Generation Function Definition**

In [25]:
# =============================================================================
# RAG V1 — BASE GENERATION FROM FROZEN RETRIEVAL — vLLM
# =============================================================================

import time


def generate_rag_from_retrieval_vllm(
    retrieval_output: dict,
    client,
    model_name: str,
    generation_config: dict,
) -> dict:
    """
    Generate an answer from a frozen retrieval output payload
    using Gemma 4 served through vLLM.

    Retrieval and prompt construction remain unchanged.
    vLLM handles chat templating, tokenization, generation,
    and response decoding on the server.
    """

    # -------------------------------------------------------------------------
    # 1. Extract retrieval payload
    # -------------------------------------------------------------------------

    query = retrieval_output.get("query", "")
    retrieved_chunks = retrieval_output.get("results", [])

    # -------------------------------------------------------------------------
    # 2. Context Assembly
    # -------------------------------------------------------------------------

    rag_context = assemble_context(retrieved_chunks)

    # -------------------------------------------------------------------------
    # 3. Build Chat Messages
    # -------------------------------------------------------------------------

    messages = build_rag_messages(
        query=query,
        context=rag_context,
    )

    # -------------------------------------------------------------------------
    # 4. Generate using vLLM Chat Completions API
    # -------------------------------------------------------------------------

    generation_start = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        **generation_config,
    )

    generation_time_sec = time.time() - generation_start

    # -------------------------------------------------------------------------
    # 5. Extract Generated Answer
    # -------------------------------------------------------------------------

    generated_answer = (
        response.choices[0].message.content or ""
    ).strip()

    # -------------------------------------------------------------------------
    # 6. Extract vLLM Token Usage
    # -------------------------------------------------------------------------

    usage = getattr(response, "usage", None)

    if usage is not None:
        input_tokens = getattr(
            usage,
            "prompt_tokens",
            None,
        )

        output_tokens = getattr(
            usage,
            "completion_tokens",
            None,
        )

        total_tokens = getattr(
            usage,
            "total_tokens",
            None,
        )

    else:
        input_tokens = None
        output_tokens = None
        total_tokens = None

    # -------------------------------------------------------------------------
    # 7. Return Standardized RAG Payload
    # -------------------------------------------------------------------------

    return {
        "query": query,
        "answer": generated_answer,

        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,

        "generation_time_sec": generation_time_sec,

        "retrieval": retrieval_output,

        "generation_config": generation_config,
    }

In [26]:
# =============================================================================
# RAG V1 — vLLM RAG GENERATION WRAPPER
# =============================================================================

def generate_rag_vllm(
    query: str,
    client,
    model_name: str,
    generation_config: dict,
    k: int = 7,
) -> dict:
    """
    Execute end-to-end Telecom RAG generation using Gemma 4
    served through vLLM.

    Pipeline:
        Query
        ↓
        Retrieval
        ↓
        Context Assembly
        ↓
        RAG Chat Messages
        ↓
        vLLM / Gemma 4
        ↓
        Answer
    """

    # -------------------------------------------------------------------------
    # 1. Live Retrieval
    # -------------------------------------------------------------------------

    retrieval_output = retrieve(
        query,
        k=k,
    )

    # -------------------------------------------------------------------------
    # 2. Generate From Retrieved Context
    # -------------------------------------------------------------------------

    return generate_rag_from_retrieval_vllm(
        retrieval_output=retrieval_output,
        client=client,
        model_name=model_name,
        generation_config=generation_config,
    )

## **Pilot - LLM + RAG Response**

In [27]:
# =============================================================================
# RAG V1 — GEMMA 4 vLLM END-TO-END PILOT TEST
# =============================================================================

print("=" * 90)
print("RAG V1 — GEMMA 4 vLLM END-TO-END PILOT TEST")
print("=" * 90)

print(f"Test Query: {TEST_QUERY}\n")
print("Running Gemma 4 vLLM RAG Pipeline...")

try:

    # =========================================================================
    # END-TO-END RAG GENERATION
    # =========================================================================

    gemma4_rag_result = generate_rag_vllm(
        query=TEST_QUERY,
        client=client,
        model_name="google/gemma-4-12B-it-qat-w4a16-ct",
        generation_config=VLLM_GENERATION_CONFIG,
        k=7,
    )

    # =========================================================================
    # CALCULATE THROUGHPUT PERFORMANCE
    # =========================================================================

    gen_time = gemma4_rag_result["generation_time_sec"]
    out_tokens = gemma4_rag_result["output_tokens"]

    throughput = (
        out_tokens / gen_time
        if gen_time is not None and gen_time > 0
        else 0.0
    )

    # =========================================================================
    # SUMMARY TELEMETRY
    # =========================================================================

    print(
        f"\nGemma 4 vLLM complete | "
        f"Input tokens: {gemma4_rag_result['input_tokens']:,} | "
        f"Output tokens: {out_tokens:,} | "
        f"Time: {gen_time:.2f}s | "
        f"Speed: {throughput:.2f} tok/s"
    )

    # =========================================================================
    # DISPLAY RESPONSE
    # =========================================================================

    print("\n" + "=" * 90)
    print("GEMMA 4 vLLM RAG ANSWER")
    print("=" * 90)

    print(gemma4_rag_result["answer"])

    print("\n" + "=" * 90)
    print("GEMMA 4 vLLM RAG PILOT TEST COMPLETE")
    print("=" * 90)

except Exception as e:

    print(f"\n[ERROR] Pipeline execution failed: {str(e)}")

RAG V1 — GEMMA 4 vLLM END-TO-END PILOT TEST
Test Query: What are the primary responsibilities of the AMF in a 5G Standalone network?

Running Gemma 4 vLLM RAG Pipeline...

Gemma 4 vLLM complete | Input tokens: 5,426 | Output tokens: 403 | Time: 17.70s | Speed: 22.76 tok/s

GEMMA 4 vLLM RAG ANSWER
The AMF (Access and Mobility Management Function) is the key control-node for the 5G access network, overseeing all signaling not specific to User Data, such as mobility or security. Its primary responsibilities include:

**Core Management Tasks**
*   **Registration and Mobility:** Performs Registration Management, Mobility Management control (including subscription and policies), and support for intra-system and inter-system mobility.
*   **Reachability:** Manages Idle mode UE Reachability, including the control and execution of paging retransmission.
*   **Security:** Handles Non-Access Stratum (NAS) signaling termination and security, as well as Access Stratum (AS) Security control.
*   **A

# **5. Telecom Benchmark Evaluation**

## **Gemma 4 + RAG Inference**

### **Track 1 Inference**

In [28]:
# =============================================================================
# RAG V1 — TRACK 1 BENCHMARK | GEMMA 4 + vLLM + TELECOM RAG
# LIVE RETRIEVAL
# =============================================================================

import json
import time
from datetime import datetime, timezone


# =============================================================================
# CONFIGURATION
# =============================================================================

TRACK1_K = 7

VLLM_MODEL = "google/gemma-4-12B-it-qat-w4a16-ct"

OUTPUT_FILE = (
    f"track1_gemma4_vllm_rag_results_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
)

track1_gemma4_results = []
track1_start = time.time()


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("RAG V1 — TRACK 1 | GEMMA 4 + vLLM + TELECOM RAG")
print("LIVE RETRIEVAL")
print("=" * 90)

print(f"Questions       : {len(benchmark_questions)}")
print(f"Retriever K     : {TRACK1_K}")
print(f"Model           : {VLLM_MODEL}")
print("=" * 90)


# =============================================================================
# EXECUTE ALL QUESTIONS
# =============================================================================

for index, item in enumerate(benchmark_questions, start=1):

    question_id = item["id"]
    category = item["category"]
    question = item["question"]

    print(
        f"\n[{index:02d}/{len(benchmark_questions):02d}] "
        f"{question_id} | {category}"
    )

    start_time = time.time()

    # =========================================================================
    # vLLM RAG INFERENCE
    # =========================================================================

    try:

        result = generate_rag_vllm(
            query=question,
            client=client,
            model_name=VLLM_MODEL,
            generation_config=VLLM_GENERATION_CONFIG,
            k=TRACK1_K,
        )

        # =====================================================================
        # SUCCESS RECORD
        # =====================================================================

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),

            "retriever": {
                "version": "V1",
                "k": TRACK1_K,
            },

            "model": {
                "name": VLLM_MODEL,
                "backend": "vLLM",
            },

            "status": "PASS",

            "answer": result["answer"],

            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "total_tokens": result["total_tokens"],

            "generation_time_sec": result["generation_time_sec"],

            "retrieval": result["retrieval"],

            "generation_config": result["generation_config"],

            "error": None,

            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        # =====================================================================
        # PERFORMANCE TELEMETRY
        # =====================================================================

        output_tokens = result["output_tokens"]
        generation_time = result["generation_time_sec"]

        throughput = (
            output_tokens / generation_time
            if output_tokens is not None and generation_time > 0
            else None
        )

        if throughput is not None:
            record["output_tokens_per_sec"] = throughput

        print(
            f"  PASS | "
            f"Input: {result['input_tokens']:,} | "
            f"Output: {result['output_tokens']:,} | "
            f"Time: {generation_time:.2f} sec"
            + (
                f" | Speed: {throughput:.2f} tok/s"
                if throughput is not None
                else ""
            )
        )

    # =========================================================================
    # ERROR HANDLING
    # =========================================================================

    except Exception as exc:

        elapsed = time.time() - start_time

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),

            "retriever": {
                "version": "V1",
                "k": TRACK1_K,
            },

            "model": {
                "name": VLLM_MODEL,
                "backend": "vLLM",
            },

            "status": "FAIL",

            "answer": None,

            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,

            "generation_time_sec": elapsed,

            "retrieval": None,

            "generation_config": VLLM_GENERATION_CONFIG,

            "error": f"{type(exc).__name__}: {exc}",

            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  FAIL | "
            f"{type(exc).__name__}: {exc}"
        )

    # =========================================================================
    # STORE RESULT
    # =========================================================================

    track1_gemma4_results.append(record)

    # =========================================================================
    # INCREMENTAL CHECKPOINT SAVE
    # =========================================================================

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(
            track1_gemma4_results,
            f,
            indent=2,
            ensure_ascii=False,
        )


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track1_elapsed = time.time() - track1_start

track1_pass = sum(
    r["status"] == "PASS"
    for r in track1_gemma4_results
)

track1_fail = sum(
    r["status"] == "FAIL"
    for r in track1_gemma4_results
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("RAG V1 — TRACK 1 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions)}")
print(f"Captured           : {len(track1_gemma4_results)}")
print(f"PASS               : {track1_pass}")
print(f"FAIL               : {track1_fail}")
print(f"Runtime            : {track1_elapsed / 60:.2f} min")
print(f"Model              : {VLLM_MODEL}")
print(f"Backend            : vLLM")
print(f"Saved payload to   : {OUTPUT_FILE}")

print("=" * 90)

RAG V1 — TRACK 1 | GEMMA 4 + vLLM + TELECOM RAG
LIVE RETRIEVAL
Questions       : 20
Retriever K     : 7
Model           : google/gemma-4-12B-it-qat-w4a16-ct

[01/20] Q01 | 5G Core
  PASS | Input: 5,426 | Output: 403 | Time: 14.69 sec | Speed: 27.44 tok/s

[02/20] Q02 | 5G Core
  PASS | Input: 5,615 | Output: 458 | Time: 19.77 sec | Speed: 23.17 tok/s

[03/20] Q03 | 5G RAN
  PASS | Input: 5,392 | Output: 277 | Time: 13.00 sec | Speed: 21.31 tok/s

[04/20] Q04 | 5G RAN
  PASS | Input: 5,907 | Output: 397 | Time: 17.67 sec | Speed: 22.46 tok/s

[05/20] Q05 | 5G SA Procedures
  PASS | Input: 6,705 | Output: 548 | Time: 23.73 sec | Speed: 23.10 tok/s

[06/20] Q06 | 5G SA Procedures
  PASS | Input: 6,619 | Output: 546 | Time: 23.60 sec | Speed: 23.13 tok/s

[07/20] Q07 | Open RAN
  PASS | Input: 5,176 | Output: 538 | Time: 22.30 sec | Speed: 24.12 tok/s

[08/20] Q08 | Open RAN
  PASS | Input: 3,853 | Output: 413 | Time: 16.92 sec | Speed: 24.41 tok/s

[09/20] Q09 | Cloud-Native Telecom
  PAS

### **Track 2 Inference**

In [29]:
# =============================================================================
# RAG V1 — TRACK 2 BENCHMARK | GEMMA 4 + vLLM + TELECOM RAG
# LIVE RETRIEVAL
# =============================================================================

import json
import time
from datetime import datetime, timezone


# =============================================================================
# CONFIGURATION
# =============================================================================

TRACK2_K = 7

VLLM_MODEL = "google/gemma-4-12B-it-qat-w4a16-ct"

OUTPUT_FILE = (
    f"track2_gemma4_vllm_rag_results_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
)

track2_gemma4_results = []
track2_start = time.time()


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("RAG V1 — TRACK 2 | GEMMA 4 + vLLM + TELECOM RAG")
print("LIVE RETRIEVAL")
print("=" * 90)

print(f"Questions       : {len(benchmark_questions_track2)}")
print(f"Retriever K     : {TRACK2_K}")
print(f"Model           : {VLLM_MODEL}")
print("=" * 90)


# =============================================================================
# EXECUTE BENCHMARK
# =============================================================================

for index, item in enumerate(benchmark_questions_track2, start=1):

    question_id = item["evaluation_id"]
    benchmark = item["benchmark"]
    question = item["question"]
    choices = item.get("choices")

    # =========================================================================
    # BUILD MODEL QUERY
    # =========================================================================

    # Prevent target-answer leakage while presenting all available choices
    if choices:
        choices_text = "\n".join(
            str(choice)
            for choice in choices
        )

        model_query = (
            f"{question}\n\n"
            f"Choices:\n"
            f"{choices_text}"
        )

    else:
        model_query = question

    print(
        f"\n[{index:02d}/{len(benchmark_questions_track2):02d}] "
        f"{question_id} | {benchmark}"
    )

    start_time = time.time()

    # =========================================================================
    # vLLM RAG INFERENCE
    # =========================================================================

    try:

        result = generate_rag_vllm(
            query=model_query,
            client=client,
            model_name=VLLM_MODEL,
            generation_config=VLLM_GENERATION_CONFIG,
            k=TRACK2_K,
        )

        # =====================================================================
        # PERFORMANCE TELEMETRY
        # =====================================================================

        output_tokens = result["output_tokens"]
        generation_time = result["generation_time_sec"]

        throughput = (
            output_tokens / generation_time
            if output_tokens is not None and generation_time > 0
            else None
        )

        # =====================================================================
        # SUCCESS RECORD
        # =====================================================================

        record = {
            "question_id": question_id,
            "benchmark": benchmark,

            "question": question,
            "choices": choices,

            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get(
                "candidate_selection_score"
            ),

            "retriever": {
                "version": "V1",
                "k": TRACK2_K,
            },

            "model": {
                "name": VLLM_MODEL,
                "backend": "vLLM",
            },

            "status": "PASS",

            "answer": result["answer"],

            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "total_tokens": result["total_tokens"],

            "generation_time_sec": result["generation_time_sec"],

            "retrieval": result["retrieval"],

            "generation_config": result["generation_config"],

            "output_tokens_per_sec": throughput,

            "error": None,

            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        # =====================================================================
        # CONSOLE TELEMETRY
        # =====================================================================

        print(
            f"  PASS | "
            f"Input: {result['input_tokens']:,} | "
            f"Output: {result['output_tokens']:,} | "
            f"Time: {generation_time:.2f} sec"
            + (
                f" | Speed: {throughput:.2f} tok/s"
                if throughput is not None
                else ""
            )
        )

    # =========================================================================
    # ERROR HANDLING
    # =========================================================================

    except Exception as exc:

        elapsed = time.time() - start_time

        record = {
            "question_id": question_id,
            "benchmark": benchmark,

            "question": question,
            "choices": choices,

            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get(
                "candidate_selection_score"
            ),

            "retriever": {
                "version": "V1",
                "k": TRACK2_K,
            },

            "model": {
                "name": VLLM_MODEL,
                "backend": "vLLM",
            },

            "status": "FAIL",

            "answer": None,

            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,

            "generation_time_sec": elapsed,

            "retrieval": None,

            "generation_config": VLLM_GENERATION_CONFIG,

            "output_tokens_per_sec": None,

            "error": f"{type(exc).__name__}: {exc}",

            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        print(
            f"  FAIL | "
            f"{type(exc).__name__}: {exc}"
        )

    # =========================================================================
    # STORE RESULT
    # =========================================================================

    track2_gemma4_results.append(record)

    # =========================================================================
    # INCREMENTAL CHECKPOINT SAVE
    # =========================================================================

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(
            track2_gemma4_results,
            f,
            indent=2,
            ensure_ascii=False,
        )


# =============================================================================
# SUMMARY
# =============================================================================

track2_elapsed = time.time() - track2_start

track2_pass = sum(
    r["status"] == "PASS"
    for r in track2_gemma4_results
)

track2_fail = sum(
    r["status"] == "FAIL"
    for r in track2_gemma4_results
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("RAG V1 — TRACK 2 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions_track2)}")
print(f"Captured           : {len(track2_gemma4_results)}")
print(f"PASS               : {track2_pass}")
print(f"FAIL               : {track2_fail}")
print(f"Runtime            : {track2_elapsed / 60:.2f} min")
print(f"Model              : {VLLM_MODEL}")
print(f"Backend            : vLLM")
print(f"Saved payload to   : {OUTPUT_FILE}")

print("=" * 90)

RAG V1 — TRACK 2 | GEMMA 4 + vLLM + TELECOM RAG
LIVE RETRIEVAL
Questions       : 32
Retriever K     : 7
Model           : google/gemma-4-12B-it-qat-w4a16-ct

[01/32] T2-01 | 3gpp_tsg
  PASS | Input: 7,764 | Output: 9 | Time: 4.74 sec | Speed: 1.90 tok/s

[02/32] T2-02 | 3gpp_tsg
  PASS | Input: 8,902 | Output: 9 | Time: 5.48 sec | Speed: 1.64 tok/s

[03/32] T2-03 | 3gpp_tsg
  PASS | Input: 8,401 | Output: 9 | Time: 5.18 sec | Speed: 1.74 tok/s

[04/32] T2-04 | 3gpp_tsg
  PASS | Input: 6,593 | Output: 9 | Time: 4.01 sec | Speed: 2.25 tok/s

[05/32] T2-05 | oranbench
  PASS | Input: 6,833 | Output: 11 | Time: 4.24 sec | Speed: 2.60 tok/s

[06/32] T2-06 | oranbench
  PASS | Input: 5,662 | Output: 47 | Time: 4.80 sec | Speed: 9.79 tok/s

[07/32] T2-07 | oranbench
  PASS | Input: 4,248 | Output: 11 | Time: 2.62 sec | Speed: 4.20 tok/s

[08/32] T2-08 | oranbench
  PASS | Input: 6,755 | Output: 6 | Time: 3.98 sec | Speed: 1.51 tok/s

[09/32] T2-09 | sixg_bench
  PASS | Input: 6,583 | Output: 